# M6d · Build the unified ML feature view

**Goal:** materialize `marts.v_ml_features_full` — the **frozen contract**
Phase 4 modelling will consume. Combines the 35-column baseline view
(`v_ml_features_driver_behavior`) with the new ~20 archive-derived columns.

**SQL file:** `sql/26_v_ml_features_full.sql` — pure `CREATE OR REPLACE VIEW`.
No data is moved; the view recomputes on every query.

**Exit criteria:**
- `SELECT * FROM marts.v_ml_features_full LIMIT 1` succeeds.
- Column count = 35 baseline + 20 new = **55 features** + 3 identity columns.
- Spot-check: every row's `harsh_events_per_100km` is finite and ≥ 0.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
from accent_fleet.db.sql_loader import load_sql
print(load_sql('26_v_ml_features_full.sql'))


## 3. Execute — create or replace the view

In [ ]:
from accent_fleet.db import run_sql_file, transaction
with transaction() as conn:
    run_sql_file(conn, '26_v_ml_features_full.sql')
print('view created.')


## 4. Inspect — schema and sample

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    cols = pd.read_sql(text('''
        SELECT column_name, data_type
          FROM information_schema.columns
         WHERE table_schema = 'marts' AND table_name = 'v_ml_features_full'
         ORDER BY ordinal_position
    '''), conn)
    n = conn.execute(text('SELECT COUNT(*) FROM marts.v_ml_features_full')).scalar_one()
    sample = pd.read_sql(text('SELECT * FROM marts.v_ml_features_full ORDER BY harsh_events_per_100km DESC LIMIT 5'), conn)

print(f'v_ml_features_full: {len(cols)} columns, {n:,} rows')
display(cols)
display(sample)

# Sanity: harsh rates are non-negative finite
import numpy as np
arr = sample[[c for c in sample.columns if c.startswith('harsh_') and 'per_100km' in c]].to_numpy()
assert np.isfinite(arr).all() and (arr >= 0).all(), 'bad harsh-rate values'
print('OK — view ready for Phase 4 modelling.')
